In [2]:
!git clone https://github.com/azrealwang/DiffMI.git
%cd /kaggle/working/DiffMI

!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q --no-deps torchjpeg
!pip install -q -r requirements.txt
!pip install -q diffusers==0.34.0

Cloning into 'DiffMI'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 165 (delta 33), reused 146 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 11.04 MiB | 37.18 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/DiffMI
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 34.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:0000:01

In [3]:
!pip uninstall -y diffusers accelerate peft transformers huggingface_hub

!pip install \
diffusers==0.25.1 \
accelerate==0.26.1 \
peft==0.8.2 \
transformers==4.37.2 \
huggingface_hub==0.20.3

Found existing installation: diffusers 0.34.0
Uninstalling diffusers-0.34.0:
  Successfully uninstalled diffusers-0.34.0
Found existing installation: accelerate 0.26.1
Uninstalling accelerate-0.26.1:
  Successfully uninstalled accelerate-0.26.1
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.10.1
Uninstalling huggingface_hub-1.10.1:
  Successfully uninstalled huggingface_hub-1.10.1
  Using cached accelerate-0.26.1-py3-none-any.whl.metadata (18 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.1 MB/s eta 0:00:00
Using cached accelerate-0.26.1-py3-none-any.whl (270 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
!pip install -q gdown
import gdown, os, zipfile, shutil
from scipy.io import loadmat

os.makedirs('latents', exist_ok=True)
gdown.download(id="15u4J5poo4kPgMvV9xyqW0SU-oDtmhU-v", output="latents/demo_dl", fuzzy=True)
print("Downloaded:", os.path.getsize("latents/demo_dl"), "bytes")

extract_dir = "latents/_extracted"
with zipfile.ZipFile("latents/demo_dl", 'r') as zf:
    zf.extractall(extract_dir)

mat_candidates, img_folder_candidates = [], []
for root, dirs, files in os.walk(extract_dir):
    dirs[:] = [d for d in dirs if d != '__MACOSX']
    if '__MACOSX' in root:
        continue
    real_files = [f for f in files if not f.startswith('._')]
    for f in real_files:
        if f.endswith('.mat'):
            mat_candidates.append((os.path.join(root, f), os.path.getsize(os.path.join(root, f))))
    if any(f.endswith(('.png', '.jpg', '.jpeg')) for f in real_files):
        img_folder_candidates.append((root, len(real_files)))

mat_file = max(mat_candidates, key=lambda x: x[1])[0]   # largest = the real one, filters out junk
img_folder = max(img_folder_candidates, key=lambda x: x[1])[0]
print("Using:", mat_file, img_folder)

shutil.copy(mat_file, "latents/demo.mat")
if os.path.exists("latents/demo"):
    shutil.rmtree("latents/demo")
shutil.copytree(img_folder, "latents/demo")

g = loadmat("latents/demo.mat")['gaussian']
print("Gaussian latents shape:", g.shape, "| Rendered images:", len(os.listdir("latents/demo")))

Downloading...
From (original): https://drive.google.com/uc?id=15u4J5poo4kPgMvV9xyqW0SU-oDtmhU-v
From (redirected): https://drive.google.com/uc?id=15u4J5poo4kPgMvV9xyqW0SU-oDtmhU-v&confirm=t&uuid=be92c9ab-d0d5-4b24-877d-b4c6b2da0301
To: /kaggle/working/DiffMI/latents/demo_dl
100%|██████████| 810M/810M [00:11<00:00, 72.8MB/s] 


Downloaded: 809781595 bytes
Using: latents/_extracted/0.999.mat latents/_extracted/0.999
Gaussian latents shape: (1000, 3, 256, 256) | Rendered images: 1000


In [5]:
base = '/kaggle/input/datasets/rajnishe/facescrub-full'
print(os.listdir(base))

['actress_faces', 'actor_faces']


In [6]:
import random
from PIL import Image

def subsample_facescrub_full(src_dirs, dst_dir, n_identities=10, n_per_identity=5, seed=0, resize=None):
    random.seed(seed)
    qualifying = []
    for src_dir in src_dirs:
        for identity in sorted(os.listdir(src_dir)):
            identity_path = os.path.join(src_dir, identity)
            if not os.path.isdir(identity_path):
                continue
            imgs = [f for f in os.listdir(identity_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(imgs) >= n_per_identity:
                qualifying.append((identity, identity_path, imgs))
    print(f"{len(qualifying)} identities qualify with >= {n_per_identity} images")
    chosen = random.sample(qualifying, n_identities)
    os.makedirs(dst_dir, exist_ok=True)
    idx = 0
    for label, (identity, identity_path, imgs) in enumerate(chosen):
        for fname in random.sample(imgs, n_per_identity):
            img = Image.open(os.path.join(identity_path, fname)).convert('RGB')
            if resize is not None:
                img = img.resize(resize)
            img.save(os.path.join(dst_dir, f"{idx:05d}_{label}.jpg"))
            idx += 1
    print(f"Wrote {idx} images across {n_identities} identities to {dst_dir}")

subsample_facescrub_full(
    src_dirs=[os.path.join(base, 'actor_faces'), os.path.join(base, 'actress_faces')],
    dst_dir='imgs/target/facescrub',
)

530 identities qualify with >= 5 images
Wrote 50 images across 10 identities to imgs/target/facescrub


In [7]:
import subprocess
log_path = 'facenet_facescrub_pilot.log'
process = subprocess.Popen(
    ["python", "-u", "DiffMI.py",
     "--input_target", "imgs/target/facescrub", "--output", "imgs/inversion/facenet_facescrub",
     "--model", "facenet", "--latent", "latents/demo",
     "--start_idx", "0", "--end_idx", "50",
     "--N", "3", "--attack", "white", "--eps", "35", "--tau_C", "0.98"],
    stdout=open(log_path, 'a'), stderr=subprocess.STDOUT,
)
print("Started, PID:", process.pid)

Started, PID: 161


In [29]:
print("Still running:", process.poll() is None)
!tail -n 30 {log_path}

Still running: False
Query Cost = 1073
Target Index: 45
Selected Candidates: [593, 939, 589]
Initial Loss = [0.59825563]
Final Loss = [0.98188037]
Query Cost = 1070
Target Index: 46
Selected Candidates: [593, 245, 217]
Initial Loss = [0.4511635]
Final Loss = [0.98061097]
Query Cost = 1073
Target Index: 47
Selected Candidates: [685, 274, 566]
Initial Loss = [0.53299534]
Final Loss = [0.98049206]
Query Cost = 1063
Target Index: 48
Selected Candidates: [685, 274, 133]
Initial Loss = [0.70697296]
Final Loss = [0.983752]
Query Cost = 1061
Target Index: 49
Selected Candidates: [685, 274, 521]
Initial Loss = [0.66581]
Final Loss = [0.98311096]
Query Cost = 1060
Attack costs 13155.837708711624s
EER = 0.05066666666764287, threshold = 0.3674318194375511
Similarity = 0.9832077622413635
ASR = 0.9599999904632568, Type I Accuracy = 1.0, Type II Accuracy = 0.949999988079071


In [30]:
import shutil
shutil.make_archive('/kaggle/working/diffmi_facenet_pilot_results', 'zip', 'imgs/inversion/facenet_facescrub')

'/kaggle/working/diffmi_facenet_pilot_results.zip'

In [31]:
import os
out_dir = 'imgs/inversion/facenet_facescrub'
print("Still in same session, files present:", os.path.exists(out_dir))
if os.path.exists(out_dir):
    files = sorted(os.listdir(out_dir))
    print(len(files), "output files")
    print(files[:5], '...', files[-5:])

Still in same session, files present: True
50 output files
['00000_0.png', '00001_0.png', '00002_0.png', '00003_0.png', '00004_0.png'] ... ['00045_9.png', '00046_9.png', '00047_9.png', '00048_9.png', '00049_9.png']


In [32]:
!grep -iE "asr|far|frr|eer" facenet_facescrub_pilot.log

EER = 0.05066666666764287, threshold = 0.3674318194375511
ASR = 0.9599999904632568, Type I Accuracy = 1.0, Type II Accuracy = 0.949999988079071


In [33]:
from utils import load_model, load_samples, predict
import torch

def gallery_accuracy(eval_model_name, gallery_dir, recon_dir, batch_size=32, topk=(1,5), device='cuda'):
    model, shape = load_model(eval_model_name)
    model = model.to(device)
    gallery_imgs, gallery_labels = load_samples(gallery_dir, shape=shape)
    gallery_feats = predict(model, gallery_imgs.to(device), batch_size)
    recon_imgs, recon_labels = load_samples(recon_dir, shape=shape)
    recon_feats = predict(model, recon_imgs.to(device), batch_size)

    correct = {k: 0 for k in topk}
    feat_dists = []
    for i in range(len(recon_feats)):
        true_label = recon_labels[i].item()
        dists = torch.norm(gallery_feats - recon_feats[i], dim=1)
        ranked = gallery_labels[torch.argsort(dists)]
        same_id = gallery_labels == true_label
        if same_id.any():
            feat_dists.append(dists[same_id].min().item())
        for k in topk:
            if true_label in ranked[:k]:
                correct[k] += 1
    n = len(recon_feats)
    return {f"Acc@{k}": correct[k]/n for k in topk}, sum(feat_dists)/len(feat_dists)

acc_crossmodel, dist_crossmodel = gallery_accuracy('arcface', gallery_dir='imgs/target/facescrub', recon_dir='imgs/inversion/facenet_facescrub')
print("Cross-model (ArcFace-evaluated) Acc@1/Acc@5:", acc_crossmodel)
print("Cross-model feature distance:", dist_crossmodel)

_, delta_face = gallery_accuracy('facenet', gallery_dir='imgs/target/facescrub', recon_dir='imgs/inversion/facenet_facescrub')
print("delta_face:", delta_face)

Downloading: "https://cdn.matix-media.net/dd/15963a78" to /root/.cache/torch/hub/checkpoints/15963a78


HTTPError: HTTP Error 521: <none>

In [ ]:
!pip install -q pytorch-fid
!python -m pytorch_fid imgs/target/facescrub imgs/inversion/facenet_facescrub

In [34]:
%cd /kaggle/working/DiffMI
!grep -n "model_urls\|iresnet100" models/insightface/iresnet.py | head -20

/kaggle/working/DiffMI
5:__all__ = ['iresnet34', 'iresnet50', 'iresnet100']
7:model_urls = {
10:    'iresnet100': 'https://cdn.matix-media.net/dd/15963a78',
155:        state_dict = load_state_dict_from_url(model_urls[arch],
171:def iresnet100(pretrained=False, progress=True, **kwargs):
172:    return _iresnet('iresnet100', IBasicBlock, [3, 13, 30, 3], pretrained, progress,


In [35]:
import urllib.request
try:
    urllib.request.urlopen("https://cdn.matix-media.net/dd/15963a78", timeout=10)
    print("Up now")
except Exception as e:
    print("Still down:", e)

Still down: HTTP Error 403: Forbidden


In [36]:
%cd /kaggle/working/DiffMI
!cat models/insightface/iresnet.py | head -20

/kaggle/working/DiffMI
import torch
from torch import nn
from torch.hub import load_state_dict_from_url

__all__ = ['iresnet34', 'iresnet50', 'iresnet100']

model_urls = {
    'iresnet34': 'https://cdn.matix-media.net/dd/09944308',
    'iresnet50': 'https://cdn.matix-media.net/dd/6fd38627',
    'iresnet100': 'https://cdn.matix-media.net/dd/15963a78',
}


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):


In [38]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="lithiumice/arcface_torch",
    filename="ms1mv3_arcface_r100_fp16.pth",
    revision="5aac4346938ae100f5145f549c4540d8bf7feb6c",
)
print("Downloaded to:", ckpt_path)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6a33cc9c-70439e574a1bed170315977b;5c886f39-96b1-4e5f-b943-ae4724866527)

Repository Not Found for url: https://huggingface.co/lithiumice/arcface_torch/resolve/5aac4346938ae100f5145f549c4540d8bf7feb6c/ms1mv3_arcface_r100_fp16.pth.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated.
Invalid username or password.

In [39]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import hashlib

ckpt_path = hf_hub_download(
    repo_id="camenduru/show",
    filename="models/arcface/ms1mv3_arcface_r100_fp16.pth",
    revision="064a379f415f674051145ec4862f54bd6a65073f",
)
print("Downloaded to:", ckpt_path)

expected_sha256 = "a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b"
sha256 = hashlib.sha256()
with open(ckpt_path, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
actual_sha256 = sha256.hexdigest()

print("Expected:", expected_sha256)
print("Actual:  ", actual_sha256)
print("MATCH" if actual_sha256 == expected_sha256 else "MISMATCH — do not load this file")

ms1mv3_arcface_r100_fp16.pth:   0%|          | 0.00/261M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/models--camenduru--show/snapshots/064a379f415f674051145ec4862f54bd6a65073f/models/arcface/ms1mv3_arcface_r100_fp16.pth
Expected: a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b
Actual:   a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b
MATCH


In [41]:
import torch
from models.insightface.iresnet import iresnet100

def load_arcface_manual(ckpt_path, device='cuda'):
    model = iresnet100(pretrained=False)
    state_dict = torch.load(ckpt_path, map_location='cpu')
    state_dict = {(k[7:] if k.startswith('module.') else k): v for k, v in state_dict.items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    model.eval().to(device)
    return model, (112, 112)

In [43]:
from utils import load_model, load_samples, predict

def gallery_accuracy(eval_model_name, gallery_dir, recon_dir, ckpt_path=None, batch_size=32, topk=(1,5), device='cuda'):
    if eval_model_name == 'arcface':
        model, shape = load_arcface_manual(ckpt_path, device)
    else:
        model, shape = load_model(eval_model_name)
        model = model.to(device)

    gallery_imgs, gallery_labels = load_samples(gallery_dir, shape=shape)
    gallery_labels = gallery_labels.to(device)
    gallery_feats = predict(model, gallery_imgs.to(device), batch_size)

    recon_imgs, recon_labels = load_samples(recon_dir, shape=shape)
    recon_labels = recon_labels.to(device)
    recon_feats = predict(model, recon_imgs.to(device), batch_size)

    correct = {k: 0 for k in topk}
    feat_dists = []
    for i in range(len(recon_feats)):
        true_label = recon_labels[i].item()
        dists = torch.norm(gallery_feats - recon_feats[i], dim=1)
        ranked = gallery_labels[torch.argsort(dists)]
        same_id = gallery_labels == true_label
        if same_id.any():
            feat_dists.append(dists[same_id].min().item())
        for k in topk:
            if true_label in ranked[:k]:
                correct[k] += 1
    n = len(recon_feats)
    return {f"Acc@{k}": correct[k]/n for k in topk}, sum(feat_dists)/len(feat_dists)

acc_crossmodel, dist_crossmodel = gallery_accuracy(
    'arcface', gallery_dir='imgs/target/facescrub', recon_dir='imgs/inversion/facenet_facescrub', ckpt_path=ckpt_path
)
print("Cross-model (ArcFace-evaluated) Acc@1/Acc@5:", acc_crossmodel)
print("Cross-model feature distance:", dist_crossmodel)

Missing keys: []
Unexpected keys: []
Cross-model (ArcFace-evaluated) Acc@1/Acc@5: {'Acc@1': 0.74, 'Acc@5': 0.9}
Cross-model feature distance: 22.605091667175294


In [45]:
import os
from PIL import Image

def make_resized_copy(src_dir, dst_dir, size=(299, 299)):
    os.makedirs(dst_dir, exist_ok=True)
    for fname in os.listdir(src_dir):
        img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
        img = img.resize(size)
        img.save(os.path.join(dst_dir, fname))
    print(f"{len(os.listdir(dst_dir))} resized images in {dst_dir}")

make_resized_copy('imgs/target/facescrub', 'imgs/target/facescrub_fid', size=(299, 299))
make_resized_copy('imgs/inversion/facenet_facescrub', 'imgs/inversion/facenet_facescrub_fid', size=(299, 299))

50 resized images in imgs/target/facescrub_fid
50 resized images in imgs/inversion/facenet_facescrub_fid


In [46]:
!python -m pytorch_fid imgs/target/facescrub_fid imgs/inversion/facenet_facescrub_fid

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.68it/s]
FID:  147.9369091209497


In [47]:
!grep -iE "asr|far|frr|eer" facenet_facescrub_run.log

grep: facenet_facescrub_run.log: No such file or directory
